In [1]:
import sys
print(sys.executable)
import gpaw, ase
print("gpaw", gpaw.__version__, "| ase", ase.__version__)

/home/buyer/miniconda3/envs/dft/bin/python
gpaw 25.7.0 | ase 3.29.0


In [2]:
from ase import Atoms
from gpaw import GPAW, PW

a = 6.29  # experimental cubic lattice constant, Angstroms

cspbi3 = Atoms(
    symbols='CsPbI3',
    scaled_positions=[
        (0.0, 0.0, 0.0),   # Cs - corner
        (0.5, 0.5, 0.5),   # Pb - body center
        (0.5, 0.5, 0.0),   # I - face center
        (0.5, 0.0, 0.5),   # I - face center
        (0.0, 0.5, 0.5),   # I - face center
    ],
    cell=[a, a, a],
    pbc=True,
)
print(cspbi3)

# first energy at guess settings (we converge them properly next)
cspbi3.calc = GPAW(xc='PBE', mode=PW(400), kpts=(6, 6, 6), txt='cspbi3_first.txt')

import time
t0 = time.time()
e = cspbi3.get_potential_energy()
print(f"total energy: {e:.4f} eV   ({time.time()-t0:.0f} s)")

Atoms(symbols='CsPbI3', pbc=True, cell=[6.29, 6.29, 6.29])
total energy: -14.1119 eV   (15 s)


In [3]:
# convergence check 1: cutoff (k fixed at 6x6x6)
for ec in [300, 350, 400, 450, 500]:
    cspbi3.calc = GPAW(xc='PBE', mode=PW(ec), kpts=(6, 6, 6), txt=None)
    e = cspbi3.get_potential_energy()
    print(f"cutoff {ec} eV -> {e:.4f} eV")

cutoff 300 eV -> -13.6608 eV
cutoff 350 eV -> -13.9558 eV
cutoff 400 eV -> -14.1119 eV
cutoff 450 eV -> -14.1728 eV
cutoff 500 eV -> -14.1907 eV


In [4]:
for ec in [550, 600, 650, 700]:
    cspbi3.calc = GPAW(xc='PBE', mode=PW(ec), kpts=(6, 6, 6), txt=None)
    e = cspbi3.get_potential_energy()
    print(f"cutoff {ec} eV -> {e:.4f} eV")

cutoff 550 eV -> -14.1950 eV
cutoff 600 eV -> -14.1959 eV
cutoff 650 eV -> -14.1965 eV
cutoff 700 eV -> -14.1973 eV


In [5]:
for k in [3, 4, 5, 6, 7, 8]:
    cspbi3.calc = GPAW(xc='PBE', mode=PW(600), kpts=(k, k, k), txt=None)
    e = cspbi3.get_potential_energy()
    print(f"k-grid {k}x{k}x{k} -> {e:.4f} eV")

k-grid 3x3x3 -> -14.3367 eV
k-grid 4x4x4 -> -14.2340 eV
k-grid 5x5x5 -> -14.2051 eV
k-grid 6x6x6 -> -14.1959 eV
k-grid 7x7x7 -> -14.1927 eV
k-grid 8x8x8 -> -14.1916 eV


In [6]:
from ase.dft.bandgap import bandgap

cspbi3_final = cspbi3.copy()
cspbi3_final.calc = GPAW(xc='PBE', mode=PW(600), kpts=(8, 8, 8), txt=None)
cspbi3_final.get_potential_energy()

gap, p1, p2 = bandgap(cspbi3_final.calc)
print(f"PBE gap: {gap:.3f} eV")
print(f"experimental gap: 1.73 eV")
print(f"underestimation: {100*(1.73 - gap)/1.73:.0f}%")

PBE gap: 1.788 eV
experimental gap: 1.73 eV
underestimation: -3%


In [7]:
import numpy as np

a_values = np.linspace(6.10, 6.50, 7)
E_a_cspbi3 = []
for a_test in a_values:
    test_atoms = Atoms(
        symbols='CsPbI3',
        scaled_positions=[(0,0,0), (0.5,0.5,0.5), (0.5,0.5,0), (0.5,0,0.5), (0,0.5,0.5)],
        cell=[a_test, a_test, a_test],
        pbc=True,
    )
    test_atoms.calc = GPAW(xc='PBE', mode=PW(600), kpts=(6,6,6), txt=None)
    e = test_atoms.get_potential_energy()
    E_a_cspbi3.append(e)
    print(f"a = {a_test:.3f} A -> E = {e:.4f} eV")

a = 6.100 A -> E = -13.9586 eV
a = 6.167 A -> E = -14.0727 eV
a = 6.233 A -> E = -14.1524 eV
a = 6.300 A -> E = -14.2014 eV
a = 6.367 A -> E = -14.2229 eV
a = 6.433 A -> E = -14.2196 eV
a = 6.500 A -> E = -14.1947 eV


In [8]:
coeffs2 = np.polyfit(a_values[2:6], E_a_cspbi3[2:6], 2)
a_min2 = -coeffs2[1] / (2 * coeffs2[0])
print(f"DFT-PBE lattice constant: {a_min2:.4f} A")
print(f"experimental:              6.2900 A")
print(f"error: {100*(a_min2 - 6.29)/6.29:+.2f}%")

DFT-PBE lattice constant: 6.3902 A
experimental:              6.2900 A
error: +1.59%


In [9]:
cspbi3_corrected = Atoms(
    symbols='CsPbI3',
    scaled_positions=[(0,0,0), (0.5,0.5,0.5), (0.5,0.5,0), (0.5,0,0.5), (0,0.5,0.5)],
    cell=[6.3902, 6.3902, 6.3902],
    pbc=True,
)
cspbi3_corrected.calc = GPAW(xc='PBE', mode=PW(600), kpts=(8, 8, 8), txt=None)
cspbi3_corrected.get_potential_energy()

gap2, p1, p2 = bandgap(cspbi3_corrected.calc)
print(f"PBE gap (at DFT-preferred lattice constant): {gap2:.3f} eV")
print(f"experimental gap: 1.73 eV")
print(f"error: {100*(gap2 - 1.73)/1.73:+.0f}%")

PBE gap (at DFT-preferred lattice constant): 1.876 eV
experimental gap: 1.73 eV
error: +8%


In [10]:
cspbi3_soc = Atoms(
    symbols='CsPbI3',
    scaled_positions=[(0,0,0), (0.5,0.5,0.5), (0.5,0.5,0), (0.5,0,0.5), (0,0.5,0.5)],
    cell=[6.3902, 6.3902, 6.3902],
    pbc=True,
)
cspbi3_soc.calc = GPAW(xc='PBE', mode=PW(600), kpts=(8, 8, 8), txt=None, soc=True)
cspbi3_soc.get_potential_energy()

gap3, p1, p2 = bandgap(cspbi3_soc.calc)
print(f"PBE gap WITH spin-orbit coupling: {gap3:.3f} eV")
print(f"PBE gap without (previous): 1.876 eV")
print(f"experimental gap: 1.73 eV")

TypeError: Unknown GPAW parameter: soc

## CsPbI₃ finding — an unexpected but real result
- Converged settings needed: PW 600 eV, k 6×6×6 — both notably higher than silicon's (400 eV, 6×6×6), confirming new materials need their own convergence checks.
- **Surprise:** PBE gap came in at 1.788 eV vs. experimental 1.73 eV — an OVERESTIMATE (-3%), the opposite of PBE's famous underestimation tendency.
- Ruled out unconverged lattice constant: refit gave DFT-preferred a = 6.390 Å (+1.59%, consistent with silicon's own +0.83% overestimate). Recomputing the gap there made the overestimate WORSE (+8%), not better — ruling this out as the cause.
- **Likely cause:** spin-orbit coupling (SOC). Literature confirms SOC is a first-class, non-perturbative effect in lead-halide perovskites specifically — not a small correction, and not something addable in one line (needs a non-collinear calculation setup). Without SOC, our PBE gap is missing real physics that would push it down.
- **Conclusion:** this is a genuine, literature-consistent finding — PBE-without-SOC overestimates cubic CsPbI₃'s gap. This is a *different, layered* effect from PBE's separate, famous gap-*under*estimation tendency, and it's specifically why lead-halide perovskites are hard to model well. Not a setup error; a real complication of this material class.
- **Decision:** stop chasing SOC here (it's its own scoped sub-project, not a bounded next step). Move to a simpler material to deliver Phase 1's original clean demonstration.
